# تدريب المُشكِّل بالحجم الكبير
فعّل بطاقة الرسوم أولًا: **وقت التشغيل ← تغيير نوع وقت التشغيل ← T4 GPU**. ثم شغّل الخلايا بالترتيب (زر ▶ على كل خلية). لا تغلق الصفحة أثناء التدريب.

In [ ]:
# ١) تنزيل الكود والبيانات
!git clone -q --depth 1 https://github.com/kaledgmal71-commits/dhad.git
!git clone -q --depth 1 https://github.com/mhashim6/Open-Hadith-Data.git
!git clone -q --depth 1 https://github.com/AliOsm/arabic-text-diacritization.git
!nvidia-smi -L

In [ ]:
# ٢) مكتبة الحوسبة على البطاقة (إن لم تكن مثبّتة)
try:
    import cupy; print('cupy', cupy.__version__)
except Exception:
    !pip -q install cupy-cuda12x
    import cupy; print('cupy', cupy.__version__)

In [ ]:
# ٣) تجهيز بيانات التدريب (نحو دقيقة)
!python3 dhad/tools/diacritizer_data.py Open-Hadith-Data hadith.npz

In [ ]:
# ٤) اختبار سريع (٢ دقيقة) للتأكد أن كل شيء يعمل على البطاقة، ويطبع سرعة الخطوات
!python3 dhad/tools/dia_train.py hadith.npz smoke --minutes 2 --d 384 --layers 6 --heads 6 --ff 1536 --T 256 --batch 64 --val arabic-text-diacritization/dataset/val.txt

In [ ]:
# ٥) التدريب الكامل (ساعتان). يحفظ نقطة كل ٢٠٠ خطوة ويطبع خطأ التحقق.
!python3 dhad/tools/dia_train.py hadith.npz dia_big --minutes 120 --d 384 --layers 6 --heads 6 --ff 1536 --T 256 --batch 64 --lr 5e-4 --val arabic-text-diacritization/dataset/val.txt

In [ ]:
# ٦) القياس النهائي على الاختبار المستقل (مرة واحدة فقط، لا تعدها)
!python3 dhad/tools/dia_eval.py dia_big arabic-text-diacritization/dataset/test.txt

In [ ]:
# ٧) تنزيل الأوزان
!zip -q dia_big.zip dia_big.npz dia_big.json dia_big.log
from google.colab import files
files.download('dia_big.zip')